# 02 - Low-confidence remasking toy sampler

**학습 목표**: 완전 마스크 상태에서 시작해 confidence가 높은 위치부터 확정하는 반복 복원을 구현합니다. 여기의 predictor는 정답을 아는 결정적 장난감 함수이므로 실제 생성 품질을 뜻하지 않습니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
import math

MASK = '[MASK]'
target = 'the image fixes most output tokens'.split()

def toy_predict(state):
    predictions = []
    for i, observed in enumerate(state):
        if observed != MASK:
            continue
        revealed_neighbors = sum(
            state[j] != MASK for j in (i - 1, i + 1) if 0 <= j < len(state)
        )
        # 문맥이 드러날수록 confidence가 높아지는 toy 규칙입니다.
        confidence = 0.55 + 0.12 * revealed_neighbors + 0.02 * (len(state) - i)
        predictions.append((confidence, i, target[i]))
    return predictions


In [ ]:
def denoise(steps):
    state = [MASK] * len(target)
    trace = []
    for step in range(steps):
        candidates = sorted(toy_predict(state), reverse=True)
        remaining_steps = steps - step
        commit_count = max(1, math.ceil(len(candidates) / remaining_steps))
        for _, position, token in candidates[:commit_count]:
            state[position] = token
        trace.append(list(state))
    # rounding 때문에 남은 위치가 있으면 마지막에 복원합니다.
    for _, position, token in toy_predict(state):
        state[position] = token
    return state, trace

result, trace = denoise(3)
for i, state in enumerate(trace, 1):
    print(f'step {i}:', ' '.join(state))
print('final:', ' '.join(result))
assert result == target

## 실험 질문

`denoise(1)`, `denoise(3)`, `denoise(len(target))`를 비교해 보세요. 실제 LLaDA에서는 predictor가 틀릴 수 있어 더 많은 step과 remasking 전략이 품질·비용 trade-off를 만듭니다.